# Práctica inicial

## Instalación y autenticación

In [ ]:
import ee

PROJECT_ID = 'vigitech-auth'

try:
    ee.Initialize(project=PROJECT_ID)
    print("GEE inicializado correctamente.")
except Exception as e:
    print(f"Error inicializando GEE: {e}. Intentando autenticación...")
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)
    print("GEE autenticado e inicializado correctamente.")

GEE inicializado correctamente.


## Asignación 1: Operaciones con números (Client side y Server side)

In [3]:
print("\n--- ASIGNACIÓN 1: OPERACIONES NUMÉRICAS ---")

# 1. Cálculos CLIENT SIDE (Python estándar)
num1_py = 10
num2_py = 3
print('Cálculos CLIENT SIDE (Python):')
print(f'Suma (10 + 3): {num1_py + num2_py}')
print(f'División (10 / 3): {num1_py / num2_py}')
print(f'Módulo (10 % 3): {num1_py % num2_py}')

# 2. Cálculos SERVER SIDE (Objetos ee.Number)
# Se necesita el objeto ee.Number para realizar operaciones con otros objetos GEE.
ee_num1 = ee.Number(10)
ee_num2 = ee.Number(3)
print('Cálculos SERVER SIDE (ee.Number):')
print(f'Suma (10 + 3): {ee_num1.add(ee_num2).getInfo()}')
print(f'División (10 / 3): {ee_num1.divide(ee_num2).getInfo()}')
print(f'Módulo (10 % 3): {ee_num1.mod(ee_num2).getInfo()}')


--- ASIGNACIÓN 1: OPERACIONES NUMÉRICAS ---
Cálculos CLIENT SIDE (Python):
Suma (10 + 3): 13
División (10 / 3): 3.3333333333333335
Módulo (10 % 3): 1
Cálculos SERVER SIDE (ee.Number):
Suma (10 + 3): 13
División (10 / 3): 3.3333333333333335
Módulo (10 % 3): 1


## Asignación 2: Creación de Listas

In [4]:
print("\n--- ASIGNACIÓN 2: CREACIÓN DE LISTAS ---")

# 1. Lista de años (2000 a 2020 con intervalo de 2) - SERVER SIDE
# Se utiliza ee.List.sequence(inicio, fin, paso)
lista_anos_server = ee.List.sequence(2000, 2020, 2)
print('Lista de Años (Server Side):', lista_anos_server.getInfo())

# 2. Lista con tres strings (client side)
lista_strings_client = ['ecológico', 'ambiental', 'productivo']
print('Lista de Strings (Client Side):', lista_strings_client)


--- ASIGNACIÓN 2: CREACIÓN DE LISTAS ---
Lista de Años (Server Side): [2000, 2002, 2004, 2006, 2008, 2010, 2012, 2014, 2016, 2018, 2020]
Lista de Strings (Client Side): ['ecológico', 'ambiental', 'productivo']


## Asignación 3: Manipulación de Listas (Arrays) - SERVER SIDE

In [5]:
print("\n--- ASIGNACIÓN 3: MANIPULACIÓN DE LISTAS ---")

lista_manipulacion = ee.List(['rojo', 'verde', 'azul'])

# get() - Acceder a un elemento por índice
print('get(1) ("verde"):', lista_manipulacion.get(1).getInfo())

# set() - Modificar un elemento por índice
lista_set = lista_manipulacion.set(0, 'amarillo')
print('set (rojo -> amarillo):', lista_set.getInfo())

# add() - Añadir un elemento al final
lista_add = lista_manipulacion.add('cian')
print('add ("cian"):', lista_add.getInfo())

# remove() - Eliminar un elemento por índice
lista_remove = lista_manipulacion.remove(2)
print('remove (índice 2):', lista_remove.getInfo())

# slice() - Obtener sub-lista
lista_slice = lista_manipulacion.slice(0, 2) # [Índice inicial, Índice final (excluido)]
print('slice (0, 2):', lista_slice.getInfo())


--- ASIGNACIÓN 3: MANIPULACIÓN DE LISTAS ---
get(1) ("verde"): verde
set (rojo -> amarillo): ['amarillo', 'verde', 'azul']
add ("cian"): ['rojo', 'verde', 'azul', 'cian']
remove (índice 2): ['rojo', 'verde', 'azul']
slice (0, 2): ['rojo', 'verde']


## Asignación 4: Manipulación de Diccionarios - SERVER SIDE

In [6]:
print("\n--- ASIGNACIÓN 4: MANIPULACIÓN DE DICCIONARIOS ---")

diccionario_ee = ee.Dictionary({
  'sensor': 'Sentinel-2',
  'resolucion': 10,
  'indice_objetivo': 'NDVI'
})

print('Diccionario original:', diccionario_ee.getInfo())

# get() - Acceder a un valor por clave
print('Valor de "resolucion" (get):', diccionario_ee.get('resolucion').getInfo())

# set() - Añadir/Modificar par clave-valor
diccionario_set = diccionario_ee.set('nubes', '< 5%')
print('Añadir "nubes":', diccionario_set.getInfo())

# keys() - Obtener una lista de claves
print('Claves (keys):', diccionario_ee.keys().getInfo())

# values() - Obtener una lista de valores
print('Valores (values):', diccionario_ee.values().getInfo())


--- ASIGNACIÓN 4: MANIPULACIÓN DE DICCIONARIOS ---
Diccionario original: {'indice_objetivo': 'NDVI', 'resolucion': 10, 'sensor': 'Sentinel-2'}
Valor de "resolucion" (get): 10
Añadir "nubes": {'indice_objetivo': 'NDVI', 'nubes': '< 5%', 'resolucion': 10, 'sensor': 'Sentinel-2'}
Claves (keys): ['indice_objetivo', 'resolucion', 'sensor']
Valores (values): ['NDVI', 10, 'Sentinel-2']


## Asignación 5: Crear su propia función

In [8]:
print("\n--- ASIGNACIÓN 5: CREAR FUNCIÓN ---")

def calcular_indice_y_limpiar(imagen_coleccion, roi):
    """
    Función que filtra una colección de imágenes, genera un composite mediano,
    calcula el NDVI y recorta el resultado a la ROI.
    """

    imagen_limpia = imagen_coleccion.filterMetadata('CLOUDY_PIXEL_PERCENTAGE', 'less_than', 5)

    imagen_mediana = imagen_limpia.median()

    ndvi = imagen_mediana.normalizedDifference(['B8', 'B4']).rename('NDVI')

    return ndvi.clip(roi)

if 'ee' in globals():
    # Se usa un buffer de 5000 metros alrededor del punto de ejemplo.
    POINT_ROI = ee.Geometry.Point(-70.0, -33.0)
    CLIPPING_ROI = POINT_ROI.buffer(5000)

    s2_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')\
        .filterDate('2024-01-01', '2024-02-01')\
        .filterBounds(CLIPPING_ROI)

    ndvi_result = calcular_indice_y_limpiar(s2_collection, CLIPPING_ROI)

    print('Función aplicada: Imagen con banda calculada (ID de la banda):', ndvi_result.bandNames().getInfo())


--- ASIGNACIÓN 5: CREAR FUNCIÓN ---
Función aplicada: Imagen con banda calculada (ID de la banda): ['NDVI']
